# Контекстный менеджер Python: пишем свой with и управляем ресурсами через ООП


Link: https://habr.com/ru/articles/1044728/

## Под капотом: Протокол контекстного менеджера

Чтобы объект мог работать с оператором with, он должен поддерживать протокол контекстного менеджера. В Python этот протокол состоит всего из двух специальных (dunder) методов: __enter__ и __exit__. Любой класс, в котором реализованы эти два метода, автоматически становится контекстным менеджером.

Давайте разберем жизненный цикл конструкции with шаг за шагом.

**1. Вызов метода __enter__** Как только интерпретатор доходит до строки с with, он берет целевой объект и немедленно вызывает его метод __enter__. Именно здесь прописывается логика подготовки: открытие файла, захват потока, старт таймера или установка сетевого соединения.

Если в конструкции используется ключевое слово as, оно захватывает ровно то, что возвращает метод __enter__. Это принципиальный момент: переменной после as присваивается не сам объект контекстного менеджера (хотя часто метод возвращает self), а именно результат работы __enter__. Если метод вернет строку или список, переменная as станет этой строкой или списком.

**2. Выполнение тела блока with** Сразу после отработки __enter__ интерпретатор переходит к выполнению основного кода, написанного с отступом внутри блока.

**3. Вызов метода __exit__** Как только выполнение тела блока завершается, интерпретатор вызывает метод __exit__. Это происходит гарантированно — независимо от того, отработал ли код штатно, встретился ли внутри return, break, continue или было выброшено критическое исключение. В этом методе концентрируется вся логика очистки: закрытие сокетов, снятие блокировок или откат незавершенных транзакций в базе данных.

## Анатомия __exit__ и управление исключениями

Метод __exit__ имеет строгую сигнатуру. Помимо стандартного self, он всегда принимает ровно три аргумента:

```code
def __exit__(self, exc_type, exc_val, exc_tb):
    # Логика очистки и обработки ошибок
```

Эти аргументы нужны для того, чтобы контекстный менеджер понимал, в каком состоянии завершился внутренний код:

Если код выполнился штатно и без ошибок, интерпретатор передаст во все три аргумента None.

Если внутри блока возникло исключение, аргументы автоматически заполнятся данными об ошибке:

 - exc_type: класс возникшего исключения (например, ValueError или KeyError).

 - exc_val: сам экземпляр исключения (объект с деталями и сообщением об ошибке).

 - exc_tb: объект traceback, содержащий трассировку стека вызовов до места падения.

По умолчанию контекстный менеджер не подавляет ошибки. Если внутри with произошел сбой, __exit__ честно выполнит свою работу по очистке ресурсов, после чего исключение полетит дальше по стеку вызовов, прерывая выполнение программы.

Однако метод __exit__ позволяет взять этот процесс под полный контроль. Если внутри __exit__ вы проанализировали переданное исключение и решили, что оно не критично (например, логика позволяет его проигнорировать), метод должен явно вернуть значение True.

Для интерпретатора возврат True из __exit__ является системным сигналом: исключение перехвачено и обработано на уровне менеджера, выбрасывать его наружу не нужно. Выполнение программы просто продолжится со следующей строки кода после блока with. Если же метод __exit__ возвращает False или ничего не возвращает (по умолчанию None), исключение пробрасывается выше стандартным образом.

In [1]:
import time

class Timer:
    def __init__(self, description="Выполнение кода"):
        # Сохраняем настройки при создании объекта
        self.description = description
        self.start_time = None

    def __enter__(self):
        # Фиксируем время старта
        self.start_time = time.time()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # Фиксируем время окончания сразу при выходе из блока
        end_time = time.time()
        
        # Вычисляем затраченное время
        elapsed_time = end_time - self.start_time
        
        # Выводим результат в консоль
        print(f"[{self.description}] завершено за {elapsed_time:.4f} сек.")
        
        # Мы ничего не возвращаем (по умолчанию вернется None), 
        # поэтому если внутри блока произойдет ошибка, исключение полетит дальше.



In [2]:
print("Начинаем работу...")

with Timer("Генерация списка и математика"):
    # Ресурсоемкая операция внутри блока
    numbers = [i * i for i in range(5_000_000)]
    sum_numbers = sum(numbers)

print("Работа окончена.")

Начинаем работу...
[Генерация списка и математика] завершено за 0.1506 сек.
Работа окончена.


In [3]:
# Вспомогательные классы-заглушки для имитации реальной БД
class MockCursor:
    def execute(self, query):
        print(f"    [SQL] {query}")

    def close(self):
        print("    Курсор закрыт.")

class MockConnection:
    def cursor(self):
        print("  Открытие курсора...")
        return MockCursor()

    def commit(self):
        print("  [SUCCESS] Транзакция ЗАФИКСИРОВАНА (commit).")

    def rollback(self):
        print("  [FAIL] Транзакция ОТКАТАЛАСЬ (rollback).")

    def close(self):
        print("  Соединение закрыто.")

# Наш контекстный менеджер
class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name
        self.connection = None
        self.cursor = None

    def __enter__(self):
        print(f"\nУстановка соединения с БД: '{self.db_name}'...")
        self.connection = MockConnection()
        self.cursor = self.connection.cursor()
        # Возвращаем курсор, чтобы именно с ним работать внутри блока with
        return self.cursor

    def __exit__(self, exc_type, exc_val, exc_tb):
        # Если exc_type не None, значит внутри блока произошла ошибка
        if exc_type is not None:
            print(f"  Обнаружена ошибка ({exc_val}). Инициируем отмену изменений.")
            self.connection.rollback()
        else:
            # Ошибок нет, можно сохранять данные
            self.connection.commit()
        
        # Блок обязательной очистки ресурсов
        self.cursor.close()
        self.connection.close()
        print("Отключение от БД завершено.")
        
        # Мы не возвращаем True, так как хотим, 
        # чтобы после отката транзакции исключение пробросилось выше 
        # и программа корректно отреагировала на сбой.


In [4]:
print("--- Сценарий 1: Успешная транзакция ---")
with DatabaseConnection('production_db') as cursor:
    cursor.execute("INSERT INTO users (name) VALUES ('Алексей')")
    cursor.execute("UPDATE stats SET total_users = total_users + 1")
    # Блок завершается штатно

print("\n--- Сценарий 2: Ошибка в процессе работы ---")
try:
    with DatabaseConnection('production_db') as cursor:
        cursor.execute("INSERT INTO users (name) VALUES ('Иван')")
        
        # Имитируем сбой: например, пришли невалидные данные
        raise ValueError("Возраст пользователя не может быть отрицательным!")
        
        # Этот код уже никогда не выполнится
        cursor.execute("UPDATE stats SET total_users = total_users + 1")
except ValueError:
    print("\nИсключение перехвачено на уровне бизнес-логики приложения.")

--- Сценарий 1: Успешная транзакция ---

Установка соединения с БД: 'production_db'...
  Открытие курсора...
    [SQL] INSERT INTO users (name) VALUES ('Алексей')
    [SQL] UPDATE stats SET total_users = total_users + 1
  [SUCCESS] Транзакция ЗАФИКСИРОВАНА (commit).
    Курсор закрыт.
  Соединение закрыто.
Отключение от БД завершено.

--- Сценарий 2: Ошибка в процессе работы ---

Установка соединения с БД: 'production_db'...
  Открытие курсора...
    [SQL] INSERT INTO users (name) VALUES ('Иван')
  Обнаружена ошибка (Возраст пользователя не может быть отрицательным!). Инициируем отмену изменений.
  [FAIL] Транзакция ОТКАТАЛАСЬ (rollback).
    Курсор закрыт.
  Соединение закрыто.
Отключение от БД завершено.

Исключение перехвачено на уровне бизнес-логики приложения.


### Бонус: Декоратор @contextmanager (Взгляд в сторону функционального подхода)

Реализация через классы дает полный контроль над процессом, но иногда создание отдельного класса ради пары строк кода инициализации и очистки выглядит избыточным. Для таких простых сценариев в стандартной библиотеке Python есть модуль contextlib и декоратор @contextmanager.

Этот инструмент позволяет создавать контекстные менеджеры на базе обычных функций-генераторов, используя ключевое слово yield.

Структура такого генератора строго повторяет жизненный цикл конструкции with:

Код до yield выполняет роль метода __enter__.

Значение, которое возвращает yield, передается в переменную после as.

Код после yield работает как __exit__ и отвечает за очистку.

Давайте перепишем наш таймер из третьей части, используя функциональный подход:

In [5]:
from contextlib import contextmanager
import time

@contextmanager
def timer_func(description="Выполнение кода"):
    # Этап __enter__
    start_time = time.time()
    
    try:
        # Передаем управление внутрь блока with
        yield 
    finally:
        # Этап __exit__
        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"[{description}] завершено за {elapsed_time:.4f} сек.")



In [6]:
with timer_func("Генерация списка"):
    numbers = [i * i for i in range(5_000_000)]

[Генерация списка] завершено за 0.1337 сек.
